# phishMe PhreshPhish Colab Workflow

Streams the pinned PhreshPhish revision, checkpoints training to Drive, and exports package-generated artifacts.

In [ ]:
from pathlib import Path

from google.colab import drive

MODE = "smoke"  # "smoke" or "full"
assert MODE in {"smoke", "full"}
LIMIT = 1000 if MODE == "smoke" else None
DRIVE_ROOT = Path("/content/drive/MyDrive/phishme-runs")
drive.mount("/content/drive")

RUN_DIR = DRIVE_ROOT / f"phresh-{MODE}"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "limit": LIMIT, "run_dir": str(RUN_DIR)})

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Bookantna/phishMe"
WORKDIR = Path("/content/phishMe")
if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
resolved_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Resolved phishMe commit:", resolved_commit)
subprocess.run(["git", "checkout", resolved_commit], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[cloud]"], check=True)

In [ ]:
from phishme import phresh

cutoff = phresh.derive_temporal_cutoff(phresh.PHRESH_REVISION)
print("PhreshPhish dataset:", phresh.PHRESH_DATASET)
print("PhreshPhish revision:", phresh.PHRESH_REVISION)
print("Temporal cutoff:", cutoff)
print("Cutoff index rule:", phresh.CUTOFF_INDEX_RULE)

In [ ]:
config = phresh.PhreshTrainConfig(
    alpha=1e-4,
    batch_size=2048 if MODE == "full" else 256,
    seed=42,
    include_dom=True,
    split="train",
    cutoff=cutoff,
)

def train_records():
    base = phresh.iter_phresh("train", revision=phresh.PHRESH_REVISION, limit=LIMIT)
    return phresh.filter_by_cutoff(base, cutoff, keep="before")

model = phresh.train_stream(train_records, CHECKPOINT_DIR, config, resume=True)
_, checkpoint = phresh.validate_checkpoint(CHECKPOINT_DIR / "checkpoint.json", config)
print({"processed_position": checkpoint["processed_position"], "class_counts": checkpoint["class_counts"]})

In [ ]:
from phishme.evaluate import metrics_report, select_threshold

if MODE == "full":
    def validation_records():
        return phresh.filter_by_cutoff(
            phresh.iter_phresh("train", revision=phresh.PHRESH_REVISION),
            cutoff,
            keep="at_or_after",
        )
    y_val, validation_scores, validation_counts = phresh.predict_stream_scores(
        model,
        validation_records,
        include_dom=True,
        batch_size=4096,
    )
    threshold = select_threshold(y_val, validation_scores)
    threshold_source = "temporal_validation"
    validation_metrics = metrics_report(y_val, validation_scores, threshold)
else:
    threshold = 0.5
    threshold_source = "smoke_only_not_validation_selected"
    validation_counts = {}
    validation_metrics = None

def test_records():
    return phresh.iter_phresh("test", revision=phresh.PHRESH_REVISION, limit=LIMIT)

y_test, test_scores, test_counts = phresh.predict_stream_scores(
    model,
    test_records,
    include_dom=True,
    batch_size=4096,
)
test_metrics = metrics_report(y_test, test_scores, threshold)
print({"threshold": threshold, "threshold_source": threshold_source})
print({"test_count": len(y_test), "test_metrics": test_metrics})

In [ ]:
from phishme.export import export_model

limitations = [
    "smoke mode is bounded and is not a benchmark" if MODE == "smoke" else "full mode still requires paired PhishLang comparison before superiority claims",
    "offline HTML may differ from live browser DOM",
    "threshold 0.5 is smoke-only and not validation-selected" if MODE == "smoke" else "threshold selected once on temporal validation",
]
model_path = RUN_DIR / "model.json"
report_path = RUN_DIR / "report.json"
manifest_path = RUN_DIR / "manifest.json"
hashes_path = RUN_DIR / "hashes.json"

export_model(
    model,
    threshold,
    True,
    {
        "dataset": phresh.PHRESH_DATASET,
        "dataset_revision": phresh.PHRESH_REVISION,
        "feature_version": phresh.FEATURE_VERSION,
        "mode": MODE,
        "cutoff": cutoff,
        "threshold_source": threshold_source,
        "checkpoint": {
            "processed_position": checkpoint["processed_position"],
            "model_joblib": checkpoint["model_joblib"],
            "model_sha256": checkpoint["model_sha256"],
        },
        "limitations": limitations,
    },
    model_path,
)
report = {
    "mode": MODE,
    "threshold": threshold,
    "threshold_source": threshold_source,
    "validation_counts": validation_counts,
    "validation_metrics": validation_metrics,
    "test_counts": test_counts,
    "test_metrics": test_metrics,
    "limitations": limitations,
}
phresh.write_json_atomically(report, report_path)
manifest = {
    "mode": MODE,
    "repo_url": REPO_URL,
    "git_commit": resolved_commit,
    "dataset": {"name": phresh.PHRESH_DATASET, "revision": phresh.PHRESH_REVISION},
    "cutoff": {"value": cutoff, "index_rule": phresh.CUTOFF_INDEX_RULE},
    "checkpoint": checkpoint,
    "limitations": limitations,
}
phresh.write_json_atomically(manifest, manifest_path)
hashes = phresh.artifact_manifest({
    "model.json": model_path,
    "report.json": report_path,
    "manifest.json": manifest_path,
    "checkpoint.json": CHECKPOINT_DIR / "checkpoint.json",
    checkpoint["model_joblib"]: CHECKPOINT_DIR / checkpoint["model_joblib"],
})
phresh.write_json_atomically(hashes, hashes_path)
print({"run_dir": str(RUN_DIR), "hashes": hashes})

In [ ]:
print("Next local smoke command:")
print("python -m phishme phresh-smoke --limit 1000 --output artifacts/phresh-smoke")
print("Next verification commands:")
print("python -m pytest tests/test_phresh.py -q")
print("node --test web/parity.test.mjs")
print("Limitations:")
for item in limitations:
    print("-", item)
if MODE == "smoke":
    print("Smoke output validates plumbing only; do not report benchmark metrics from this bounded run.")